In [1]:
import os
os.makedirs('src', exist_ok=True)
print(os.listdir('.'))

['05_comparison_analysis.ipynb', '04_bellman_ford.ipynb', '00_export_src.ipynb', '.ipynb_checkpoints', '02_dijkstra.ipynb', '01_graph_representation.ipynb', '03_prim.ipynb', 'src']


In [2]:
%%writefile src/graph.py
class Graph:
    def __init__(self):
        self.adj = {}

    def add_node(self, node):
        if node not in self.adj:
            self.adj[node] = []

    def add_edge(self, u, v, weight):
        self.add_node(u)
        self.add_node(v)
        self.adj[u].append((v, weight))

    def nodes(self):
        return list(self.adj.keys())

    def edges(self):
        result = []
        for u in self.adj:
            for v, w in self.adj[u]:
                result.append((u, v, w))
        return result

    def neighbors(self, u):
        return self.adj.get(u, [])

    def num_nodes(self):
        return len(self.adj)

    def num_edges(self):
        return sum(len(v) for v in self.adj.values())

    def __repr__(self):
        lines = []
        for u in self.adj:
            for v, w in self.adj[u]:
                lines.append(f"  {u} -> {v}  (weight={w})")
        return f"Graph({self.num_nodes()} nodes, {self.num_edges()} edges)\n" + "\n".join(lines)

Writing src/graph.py


In [3]:
%%writefile src/dijkstra.py
import heapq
from graph import Graph


def dijkstra(graph: Graph, source, track_steps=False):
    distances = {node: float('inf') for node in graph.nodes()}
    predecessors = {node: None for node in graph.nodes()}
    distances[source] = 0

    visited = set()
    pq = [(0, source)]
    steps = []

    while pq:
        dist_u, u = heapq.heappop(pq)

        if u in visited:
            continue
        visited.add(u)

        if track_steps:
            steps.append({
                "finalized_node": u,
                "distances": distances.copy(),
                "visited": visited.copy()
            })

        for v, weight in graph.neighbors(u):
            if weight < 0:
                raise ValueError(f"Dijkstra's algorithm cannot handle negative edge weight: {u}->{v} = {weight}")
            new_dist = dist_u + weight
            if new_dist < distances[v]:
                distances[v] = new_dist
                predecessors[v] = u
                heapq.heappush(pq, (new_dist, v))

    if track_steps:
        return distances, predecessors, steps
    return distances, predecessors


def reconstruct_path(predecessors, source, target):
    if predecessors.get(target) is None and target != source:
        return None
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = predecessors[node]
    path.reverse()
    return path if path[0] == source else None

Writing src/dijkstra.py


In [4]:
%%writefile src/prim.py
import heapq
from graph import Graph


def to_undirected_adjacency(graph: Graph):
    undirected = {node: {} for node in graph.nodes()}
    for u, v, w in graph.edges():
        if v not in undirected[u] or w < undirected[u][v]:
            undirected[u][v] = w
        if u not in undirected[v] or w < undirected[v][u]:
            undirected[v][u] = w
    return undirected


def prim(graph: Graph, start=None, track_steps=False):
    undirected = to_undirected_adjacency(graph)
    nodes = graph.nodes()
    if not nodes:
        return ([], 0) if not track_steps else ([], 0, [])

    start = start or nodes[0]
    visited = {start}
    mst_edges = []
    total_weight = 0
    steps = []

    pq = []
    for v, w in undirected[start].items():
        heapq.heappush(pq, (w, start, v))

    while pq and len(visited) < len(nodes):
        w, u, v = heapq.heappop(pq)
        if v in visited:
            continue

        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w

        if track_steps:
            steps.append({
                "edge_added": (u, v, w),
                "visited": visited.copy(),
                "mst_edges_so_far": mst_edges.copy()
            })

        for nxt, weight in undirected[v].items():
            if nxt not in visited:
                heapq.heappush(pq, (weight, v, nxt))

    return (mst_edges, total_weight) if not track_steps else (mst_edges, total_weight, steps)

Writing src/prim.py


In [5]:
%%writefile src/bellman_ford.py
from graph import Graph


def bellman_ford(graph: Graph, source, track_steps=False):
    distances = {node: float('inf') for node in graph.nodes()}
    predecessors = {node: None for node in graph.nodes()}
    distances[source] = 0

    all_edges = graph.edges()
    n = graph.num_nodes()
    steps = []

    for i in range(n - 1):
        updated = False
        for u, v, w in all_edges:
            if distances[u] != float('inf') and distances[u] + w < distances[v]:
                distances[v] = distances[u] + w
                predecessors[v] = u
                updated = True

        if track_steps:
            steps.append({"pass": i + 1, "distances": distances.copy()})

        if not updated:
            break

    has_negative_cycle = False
    for u, v, w in all_edges:
        if distances[u] != float('inf') and distances[u] + w < distances[v]:
            has_negative_cycle = True
            break

    if track_steps:
        return distances, predecessors, has_negative_cycle, steps
    return distances, predecessors, has_negative_cycle

Writing src/bellman_ford.py


In [6]:
import os
print(os.listdir('src'))

['dijkstra.py', 'prim.py', 'graph.py', 'bellman_ford.py']
